In [ ]:
from huggingface_hub import hf_hub_download, HfApi

token = os.environ["HF_TOKEN"]   # paste your token

repo = "google/gemma-2-2b-it"

api = HfApi()

try:
    api.repo_info(repo_id=repo, token=token)
    print("✔ Token CAN access the model.")
except Exception as e:
    print("❌ Token CANNOT access the model.")
    print(e)


In [ ]:
# Authenticate with Hugging Face (required for gated models, e.g. Llama/Gemma checkpoints)
from huggingface_hub import login
import os
login(os.environ["HF_TOKEN"])

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import precision_score, recall_score, f1_score

# ------------------------------
# 1. Load Model on GPU
# ------------------------------
model_name = "google/gemma-2-2b-it"
token = os.environ["HF_TOKEN"]   # <-- paste your HF token here

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto", 
    use_auth_token=token
)

In [ ]:
# ------------------------------
# 2. Classification Function
# ------------------------------
input_csv_path = "./data/LLaVA-Med/All_ADRs - Sheet1.csv"
output_csv_path = "./data/LLaVA-Med/Results_Gemma_ClassificationADR_151125_1.csv" 

def classify_text(text):   
    prompt = f"""
    You are an excellent clinical information classifier system. Decide whether the text describes an Adverse Drug Effect.  
    Respond strictly with "Yes" or "No". An adverse drug event is any harmful or negative experience related to ongoing medication or treatment. 

    Text: {text}

    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output_tokens = model.generate(
        **inputs,
        max_new_tokens=3,
        do_sample=False
    )

    response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    response = response.strip().split("Answer:")[-1].strip()

    # Normalize output
    if response.lower().startswith("y"):
        return "Yes"
    else:
        return "No"

In [ ]:
# df = pd.read_csv(input_csv_path)
# print(df.columns.tolist())

In [ ]:
# ------------------------------
# 3. Load CSV
# ------------------------------
df = pd.read_csv(input_csv_path)
print(df.columns.tolist())

texts = df["Preprocessed Posts"].fillna("").astype(str).tolist()
gold_labels = df["Adverse effects(Yes/No)"].str.strip()


In [ ]:
# ------------------------------
# 4. Run Classification
# ------------------------------
predictions = []

for text in texts:
    pred = classify_text(text)
    print(pred)
    predictions.append(pred)

df["Output - Adverse effects(Yes/No)"] = predictions

# ------------------------------
# 5. Compute Metrics
# ------------------------------
prec = precision_score(gold_labels, predictions, pos_label="Yes")
rec = recall_score(gold_labels, predictions, pos_label="Yes")
f1 = f1_score(gold_labels, predictions, pos_label="Yes")

# Create empty columns first
df["Cumulative Precision"] = ""
df["Cumulative Recall"] = ""
df["Cumulative F1 Score"] = ""

# Add metrics ONLY to the last row
df.loc[df.index[0], "Cumulative Precision"] = prec
df.loc[df.index[0], "Cumulative Recall"] = rec
df.loc[df.index[0], "Cumulative F1 Score"] = f1

# ------------------------------
# 6. Save Output CSV
# ------------------------------
df.to_csv(output_csv_path, index=False)

print("Done! File saved to {output_csv_path}")
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)
